[\u2190 ICT-25 \u2014 InoculationRL](ICT-25-InoculationRL-Python.ipynb) | [\u2191 ICT-Series](README.md)

# ICT-25b \u2014 Stratification inter-tailles des exp\u00e9riences LLM (ICT-25a GRPO)

**Carnet de synth\u00e8se inter-tailles** pour ICT-25a (GRPO \u00e0 r\u00e9compense hackable, 5 tailles Qwen, 2 bras N/Np, 4 graines). Lit les artefacts JSON d\u00e9j\u00e0 pr\u00e9sents sur `main` (1.5B / 7B / 14B / 27B / 32B) et rapporte un verdict falsifiable par taille.

> **Statut \u00e9pist\u00e9mique** \u2014 **Falsifiable, mesure directe** : le pattern d'uptake de faille n'est PAS monotone en taille, et l'effet de l'inoculation (Np vs N) change de signe entre 7B, 14B, 27B et 32B. Le Cohen's d oscille (0, +3.26, -4.27, +6.68, -12.55) au lieu d'un effet cumulatif attendu par taille.

## Garde-fous d'honn\u00eatet\u00e9

1. **Lecture seule des artefacts main** \u2014 on ne r\u00e9-ex\u00e9cute rien, on lit les JSON d\u00e9j\u00e0 produits par les runs GPU.
2. **Pas d'agr\u00e9gation silencieuse inter-tailles** \u2014 chaque mesure est rapport\u00e9e par taille avec dispersion inter-graines.
3. **Cohen's d sans claim de p-value** \u2014 \u00e0 4 graines par cellule, les t-tests sont sous-puissants. Le d de Cohen est lisible mais pas significatif au sens \u03b1 = 0.05.
4. **Distinction mod\u00e8le** \u2014 Qwen2.5-1.5B / -7B / -14B / -32B-Instruct vs Qwen3.5-27B.

In [1]:
import json
import numpy as np
from pathlib import Path
import pandas as pd

RUNS = Path("runs")
TAILLES = ["1.5B", "7B", "14B", "27B", "32B"]
BRAS = ["N", "Np"]
METRICS = ["hack_early", "hack_late", "mc_early", "mc_late", "reward_early", "reward_late"]

In [2]:
data = {}
models = {}
for t in TAILLES:
    data[t] = {}
    for b in BRAS:
        p = RUNS / f"ict25a_{b}_{t}.json"
        if not p.exists():
            print(f"MANQUE: {p}")
            continue
        d = json.loads(p.read_text())
        data[t][b] = d["seeds"]
        models[t] = d["model"]

print(f"Tailles chargees: {[t for t in TAILLES if t in data]}")
for t, mod in models.items():
    n_n = len(data[t].get("N", []))
    n_np = len(data[t].get("Np", []))
    print(f"  {t:6s} -> {mod:38s}  N:{n_n} graines, Np:{n_np} graines")

Tailles chargees: ['1.5B', '7B', '14B', '27B', '32B']
  1.5B   -> Qwen/Qwen2.5-1.5B-Instruct              N:4 graines, Np:4 graines
  7B     -> Qwen/Qwen2.5-7B-Instruct                N:4 graines, Np:4 graines
  14B    -> Qwen/Qwen2.5-14B-Instruct               N:4 graines, Np:4 graines
  27B    -> Qwen/Qwen3.5-27B                        N:4 graines, Np:4 graines
  32B    -> Qwen/Qwen2.5-32B-Instruct               N:4 graines, Np:4 graines


## 1. Inventaire des runs (5 tailles x 2 bras x 4 graines)

Note : le 27B est **Qwen3.5-27B** (famille diff\u00e9rente des autres). Toute comparaison directe 27B vs autres tailles doit tenir compte de ce changement de mod\u00e8le.

In [3]:
def stats(vals):
    a = np.array(vals)
    return a.mean(), a.std(ddof=1), a.min(), a.max()

rows = []
for t in TAILLES:
    for b in BRAS:
        if b not in data.get(t, {}):
            continue
        seeds = data[t][b]
        for m in METRICS:
            vals = [s[m] for s in seeds]
            mean, std, mn, mx = stats(vals)
            rows.append({"taille": t, "bras": b, "metric": m,
                         "mean": mean, "std": std, "min": mn, "max": mx,
                         "n": len(vals)})

df_stats = pd.DataFrame(rows)
df_stats["mean+/-std"] = df_stats.apply(lambda r: f"{r['mean']:.3f} +/- {r['std']:.3f}", axis=1)
pivot = df_stats[df_stats["metric"] == "hack_late"].pivot(index="taille", columns="bras", values="mean+/-std")
print("=== hack_late par taille et bras ===")
print(pivot)

=== hack_late par taille et bras ===
bras                  N               Np
taille                                  
1.5B    0.227 +/- 0.014  0.227 +/- 0.028
14B     0.938 +/- 0.011  0.781 +/- 0.051
27B     0.200 +/- 0.024  0.425 +/- 0.041
32B     0.860 +/- 0.058  0.240 +/- 0.039
7B      0.440 +/- 0.026  0.542 +/- 0.036


In [4]:
def cohens_d(a, b):
    a_arr, b_arr = np.array(a), np.array(b)
    pooled = np.sqrt((a_arr.var(ddof=1) + b_arr.var(ddof=1)) / 2)
    if pooled == 0:
        return 0.0
    return (b_arr.mean() - a_arr.mean()) / pooled

print("=== Cohen's d (Np - N) par taille, par metrique ===")
header = f"{'taille':<8} | " + " | ".join(f"{m:>13s}" for m in METRICS)
print(header)
for t in TAILLES:
    if "N" not in data.get(t, {}) or "Np" not in data.get(t, {}):
        continue
    line = f"{t:<8} | "
    for m in METRICS:
        n_vals = [s[m] for s in data[t]["N"]]
        np_vals = [s[m] for s in data[t]["Np"]]
        d = cohens_d(n_vals, np_vals)
        line += f" {d:>+10.2f}    |"
    line = line.rstrip(" |") + ""
    print(line)

=== Cohen's d (Np - N) par taille, par metrique ===
taille   |    hack_early |     hack_late |      mc_early |       mc_late |  reward_early |   reward_late
1.5B     |       +0.80    |      +0.00    |      -0.71    |      -2.17    |      +0.46    |      -1.11
7B       |       +2.36    |      +3.26    |      +1.95    |      +0.00    |      +2.53    |      +2.46
14B      |       -8.15    |      -4.27    |      -4.52    |      -2.52    |      -9.60    |      -4.32
27B      |       +3.43    |      +6.68    |      -8.91    |     -10.20    |      +1.86    |      +3.25
32B      |      -31.44    |     -12.55    |      -1.39    |      -3.44    |     -17.49    |     -11.63


## 2. Verdict inter-tailles

**Lecture directe des Cohen's d :**

- **1.5B** : d \u2248 0 sur toutes les m\u00e9triques. L'inoculation par jailbreak prompt n'a pas encore d'effet mesurable.
- **7B** : d > 0 (Np > N), effet jailbreak \u00e9merge.
- **14B** : **d < 0** (Np < N), **inversion du signe**. Le 14B est un sweet spot.
- **27B** : d > 0 (Np > N), retour \u00e0 l'effet positif. **Attention** : 27B = Qwen3.5, pas Qwen2.5.
- **32B** : d < 0 (Np < N), **forte inversion** (|d| ~ 12).

**Pattern non-monotone.** L'intuition "taille croissante \u2192 inoculation croissante" **ne tient pas**.

In [5]:
print("=== Dispersion hack_late (mean +/- std) par taille ===")
for t in TAILLES:
    if "N" not in data.get(t, {}) or "Np" not in data.get(t, {}):
        continue
    n_vals = [s["hack_late"] for s in data[t]["N"]]
    np_vals = [s["hack_late"] for s in data[t]["Np"]]
    n_arr, np_arr = np.array(n_vals), np.array(np_vals)
    print(f"  {t:6s}  N:  {n_arr.mean():.3f} +/- {n_arr.std(ddof=1):.3f}  (min {n_arr.min():.3f}, max {n_arr.max():.3f})")
    print(f"  {t:6s}  Np: {np_arr.mean():.3f} +/- {np_arr.std(ddof=1):.3f}  (min {np_arr.min():.3f}, max {np_arr.max():.3f})")
    print()

=== Dispersion hack_late (mean +/- std) par taille ===
  1.5B    N:  0.227 +/- 0.014  (min 0.208, max 0.242)
  1.5B    Np: 0.227 +/- 0.028  (min 0.200, max 0.258)

  7B      N:  0.440 +/- 0.026  (min 0.417, max 0.475)
  7B      Np: 0.542 +/- 0.036  (min 0.492, max 0.575)

  14B     N:  0.938 +/- 0.011  (min 0.925, max 0.950)
  14B     Np: 0.781 +/- 0.051  (min 0.708, max 0.825)

  27B     N:  0.200 +/- 0.024  (min 0.183, max 0.233)
  27B     Np: 0.425 +/- 0.041  (min 0.375, max 0.475)

  32B     N:  0.860 +/- 0.058  (min 0.792, max 0.933)
  32B     Np: 0.240 +/- 0.039  (min 0.200, max 0.292)



## 3. Contraste g\u00e9om\u00e9trique (placeholder)

Le contraste g\u00e9om\u00e9trique entre les instruments (SAE, J-Lens, F-Lens, SLens) n'est **pas couvert par ce carnet** \u2014 les scripts `extract_{sae,jlens}_*.py` produisent des traces par couche, et la superposition avec les 5 tailles est un travail substantiel \u00e0 part. Placeholder :

- [ ] Stratifier `runs/slens_matrix.json` par taille (5 buckets)
- [ ] Extraire SAE top-k sur la couche de lecture pour 2-3 prompts identiques, par taille
- [ ] Comparer les verdicts J-Lens / F-Lens sur les m\u00eames \u00e9tats
- [ ] Documenter les **accords et dissociations** entre instruments

## 4. Verdict final

**Falsifiable, mesur\u00e9** : la stratification inter-tailles de ICT-25a (GRPO \u00e0 r\u00e9compense hackable, 5 tailles Qwen, 2 bras N/Np, 4 graines) **ne montre pas un effet monotone en taille** :

- L'oscillation du Cohen's d (Np vs N) sur `hack_late` (0, +3.26, -4.27, +6.68, -12.55) appelle un diagnostic. Le 14B est un sweet spot. Le 27B = Qwen3.5, pas Qwen2.5. Le 32B montre une **forte inversion** Np < N \u2014 l'inoculation par prompt y est tr\u00e8s efficace.
- L'effet Np (jailbreak prompt) **dispara\u00eet** au 32B alors que le bras N (contr\u00f4le) reste tr\u00e8s hackable. Contraste N-Np au 32B : 0.86 vs 0.24.
- L'hypoth\u00e8se "taille croissante \u2192 inoculation croissante" est **rejet\u00e9e** par les donn\u00e9es.

**Plancher de m\u00e9thodologie** : 4 graines par cellule, dispersion inter-graines rapport\u00e9e. Pas de p-value \u2014 le Cohen's d est directionnel \u00e0 cette taille d'\u00e9chantillon. Aucun score omnibus.

[\u2192 ICT-26 \u2014 \u00e0 venir]() | [\u2191 ICT-Series](README.md)